# Fixed three-phase MM regularization with an L2 anchor

**Experiment:** frozen pretrained baseline U-Net → three-class softmax → fixed Modica–Mortola + separator + boundary energy + L2 anchor → 8-connected nuclear components. **No learning and no watershed.**

Outputs: paired semantic Dice, instance F1 (IoU ≥ 0.5), panoptic quality, merge/split rates at multiple iteration counts and anchor strengths. Scores are computed for **2D slices**, not 3D nuclear objects.

The supplied energy implementation is used unchanged. The anchor is implemented in `anchored_refinement.py` and the U-Net model in `UNet_baseline.py`.

**Important:** The hand-tuned energy coefficients were not included in the handoff; the values below are illustrative starting values. Replace them with your empirically successful values before interpreting negative findings.

## 1. Setup

Download the accompanying `anchored_mm_colab_bundle.zip`, open this notebook in Colab, and upload the ZIP in the next cell. It contains all the project's required experiment files. GPU runtime is recommended (**Runtime → Change runtime type → T4 GPU**).

In [ ]:
%pip -q install psfmodels scipy scikit-image numba matplotlib pandas

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')

uploaded = files.upload()  # select anchored_mm_colab_bundle.zip
import zipfile, pathlib, sys
zip_path = next((pathlib.Path(name) for name in uploaded if name.endswith('.zip')), None)
if zip_path is None:
    raise ValueError('Upload anchored_mm_colab_bundle.zip')
with zipfile.ZipFile(zip_path) as zf:
    zf.extractall('/content')
PROJECT_SOURCE = pathlib.Path('/content/anchored_mm_colab')
assert (PROJECT_SOURCE / 'mm_fixed_compatible.py').exists()
sys.path.insert(0, str(PROJECT_SOURCE))
print('Experiment source:', PROJECT_SOURCE)

## 2. Inspect the baseline checkpoint

The path you provided may refer to a **directory** or to a checkpoint file. If the directory has multiple checkpoints, choose the exact one; we intentionally do not silently guess which is best. The loader performs strict matching against the attached U-Net architecture.

In [ ]:
from pathlib import Path
BASELINE_PATH = Path('/content/drive/MyDrive/Colab Notebooks/proj_26/checkpoints_unet_baseline_v1_boosted_ce')
print('Exists:', BASELINE_PATH.exists(), '| Directory:', BASELINE_PATH.is_dir())
if BASELINE_PATH.is_dir():
    for p in sorted(BASELINE_PATH.rglob('*')):
        if p.is_file() and p.suffix.lower() in ('.pt', '.pth', '.ckpt', '.bin'):
            print(p)
# If there are multiple files, replace BASELINE_PATH with the chosen full filename.

## 3. Experiment configuration

We use your dataset settings (`N=192`, `z_slices=16`, `z_ratio=2`, `n_seeds=45`, `class_probs=(2/5,1/5,2/5)`, domain `a`). We **do not** use the accidentally repeated `dataset_train` from the sample validation loader. Seed `100000` avoids overlap with the training generator's `seed=0` and indices 0–999.

**Data leakage note:** if your U-Net was trained on seeds other than 0, verify that this evaluation seed is genuinely held out. For a final test, use unseen random seeds/domains. Set `n_volumes=50` to use your proposed validation volume count.

**Compute note:** 400 iterations × 4 anchor strengths × 16 slices × several volumes is expensive. Try the pilot cell first.

In [ ]:
import torch
from run_experiment import RunConfig, run

cfg = RunConfig(
    checkpoint=str(BASELINE_PATH),
    output_dir='/content/drive/MyDrive/Colab Notebooks/proj_26/anchored_mm_results',
    device='cuda' if torch.cuda.is_available() else 'cpu',
    seed=100000,              # disjoint from seed=0, indices 0..999
    n_volumes=4,             # pilot: 1; full validation: 50
    volume_N=192,
    z_slices=16,
    z_ratio=2,
    n_seeds=45,
    class_probs=(2/5, 1/5, 2/5),
    domain='a',
    anchor_values=(0.0, 0.02, 0.1, 0.5),
    steps=400,
    checkpoints=(0, 25, 50, 100, 200, 300, 400),
    step_size=0.02,
    backtracking=False,      # optional True: test total anchored energy decrease
    inference_batch_size=4,
    refine_batch_size=2,
    energy=dict(
        lam=0.2, epsilon=0.5, well_cubic_weight=1.0,
        barrier_weight=0.1, barrier_width=5, nuclear_band_width=2,
        barrier_length=13, barrier_longitudinal_sigma=3.5,
        barrier_orientations=8,
        boundary_weight=0.5, boundary_middle_width=3,
        boundary_outer_width=2, boundary_orientations=16,
        learn_strengths=False, learn_residuals=False,
    ),
)
print('Device:', cfg.device)

### Optional pilot

A single volume with 25 steps verifies checkpoint compatibility, data generation, and evaluation quickly. Outputs go into a separate folder. Then run the main experiment below. (Even the pilot requires the synthetic data generator and pretrained weights.)

In [ ]:
from dataclasses import replace
RUN_PILOT = False
if RUN_PILOT:
    pilot = replace(cfg, n_volumes=1, steps=25, checkpoints=(0, 5, 10, 25),
                    anchor_values=(0.0, 0.1),
                    output_dir=cfg.output_dir + '_pilot')
    pilot_summary = run(pilot)

## 4. Run anchored refinement and score every slice

Each U-Net prediction is generated once per volume. For every anchor strength, the energy is applied for up to 400 steps, saving the requested intermediate snapshots. Original probabilities remain fixed as the anchor reference. The existing **reduced-coordinate projection** is retained.

For both ground truth and predictions, the nuclear mask is `class == 1`, with the prediction class taken by `argmax` of the softmax probabilities. Components use **8-connectivity**, with no watershed and no minimum-size filtering.

Metrics: per-class semantic Dice, 8-CC instance F1 (one-to-one IoU matching at 0.5), PQ, merge and split counts/rates. Merge/split overlap criteria are documented in `instance_metrics.py`. Ground-truth overlap pixels marked `-100` are excluded.

In [ ]:
results = run(cfg)
print('Finished:', len(results), 'parameter / horizon combinations')

## 5. View results

Saved artifacts in Drive: `settings.json`, `per_slice.csv`, `summary.csv`, `curves.png`, and one `preview_mu_*.png` per anchor (for the selected volume). `summary.csv` contains macro per-slice Dice/F1/PQ and pooled instance F1.

**Interpretation:** `mu=0` is the unanchored control; `iteration=0` is the same raw U-Net baseline in every anchor group. In addition to improvements in instance F1, check false-positive components and merge/split rates. Choose any regularization settings using a validation set, then report final metrics on a separate test seed.

In [ ]:
import pandas as pd
from IPython.display import display, Image
out = Path(cfg.output_dir)
df = pd.read_csv(out / 'summary.csv')
display(df[['anchor_mu','iteration','dice_1','dice_2','micro_instance_f1','merge_rate','split_rate','pred_instances','gt_instances']])
display(Image(filename=str(out / 'curves.png')))

In [ ]:
from IPython.display import Image, display
for mu in cfg.anchor_values:
    p = Path(cfg.output_dir) / f'preview_mu_{mu:g}.png'
    if p.exists():
        print('Anchor mu =', mu)
        display(Image(filename=str(p)))

## Notes for extending the experiment

- `unet_inference.py`: checkpoint resolution, strict U-Net loading, frozen softmax inference.
- `anchored_refinement.py`: L2 anchor, analytic reduced-coordinate gradient, projected solver, optional backtracking on total energy.
- `instance_metrics.py`: 8-connected component labeling, semantic Dice, IoU matching, instance F1, PQ, merge/split counts.
- `run_experiment.py`: data construction, paired sweeps, CSV aggregation, images.
- `mm_fixed_compatible.py`: unchanged original energy and kernel definitions.

If you see instability with the fixed step size, turn on `backtracking=True` or lower `step_size`. With backtracking, the full anchored objective must decrease; this is slower. **A one-pixel bridge can merge components** under 8-connectivity, so visually inspect failures alongside the instance scores. Instance labels are formed independently on each 2D slice, even though the data generator is volumetric.